# Trực quan hóa kết quả

Thay thế ba cell "VISUALIZE" của notebook cũ, vốn viết cho một API chưa từng tồn
tại (`from src.config import cfg`, `cfg.OUT`, `cfg.META_DIM`,
`ClassifierHead(768, ...)`, `student(t, return_attn=True)`) và không chạy được.

Notebook này dùng đúng API hiện hành. Chạy sau khi đã có checkpoint.


In [ ]:
# 0. Setup
import sys, logging
from pathlib import Path

PROJECT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(PROJECT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from knee_mri.config import load_config
from knee_mri.constants import LABELS
from knee_mri.data.catalog import StudyCatalog
from knee_mri.utils.logging import setup_logging

setup_logging(logging.WARNING)
cfg = load_config()
print('môi trường:', cfg.env, '| dataset:', cfg.paths.data_root)
%matplotlib inline


## 1. Phân bố xác suất dự đoán

In [ ]:
submission_path = cfg.paths.predictions / 'submission.csv'
if not submission_path.is_file():
    print('Chưa có submission.csv — chạy scripts/predict.py trước.')
else:
    submission = pd.read_csv(submission_path)
    probabilities = submission[list(LABELS)].astype(float).values
    print('Số study:', len(submission))
    print('Xác suất trung bình mỗi nhãn:')
    for name, value in zip(LABELS, probabilities.mean(0)):
        print(f'  {name:<18s} {value:.3f}')

    fig, axes = plt.subplots(3, 4, figsize=(15, 8))
    for ax, name, column in zip(axes.ravel(), LABELS, probabilities.T):
        ax.hist(column, bins=20, color='#4c72b0')
        ax.set_title(name, fontsize=10)
        ax.set_xlim(0, 1)
    fig.suptitle('Phân bố xác suất dự đoán theo nhãn')
    fig.tight_layout()
    plt.show()


## 2. CAM overlay

Kiểm chứng student "nhìn" vào đâu. CAM chỉ dựng được vì
`AttentionBlock` thực sự trả về trọng số attention — bản cũ dùng forward hook
trên `nn.TransformerEncoderLayer`, vốn luôn trả `None`, nên tính năng này chưa
bao giờ chạy.


In [ ]:
from knee_mri.data.normalize import to_unit_range
from knee_mri.data.series_selection import representative_series
from knee_mri.data.volume import build_volume
from knee_mri.explain.cam import attention_rollout, attention_to_volume
from knee_mri.explain.overlay import cam_mask_overlap
from knee_mri.models.heads import ClassifierHead, make_metadata_vector
from knee_mri.models.student import ViT3DStudent
from knee_mri.training.checkpoint import BEST_NAME, load_for_inference
from knee_mri.utils.mask import load_mask

device = 'cuda' if torch.cuda.is_available() else 'cpu'
checkpoint = cfg.paths.checkpoints / BEST_NAME

if not checkpoint.is_file():
    print('Chưa có checkpoint — chạy scripts/train.py trước.')
else:
    catalog = StudyCatalog.from_csv(cfg.paths.train_csv, cfg.paths.train_series_csv)
    student, classifier = ViT3DStudent(cfg), ClassifierHead(cfg)
    load_for_inference(checkpoint, {'student': student, 'classifier': classifier}, device=device)

    study_uid = catalog.studies_with_series()[0]
    series = representative_series(catalog, study_uid, cfg.data)
    series_dir = cfg.paths.series_dir(study_uid, series.series_uid)
    mask = load_mask(study_uid, series.series_uid, cfg.paths.masks)
    volume = build_volume(series_dir, cfg.data, mask=mask)

    with torch.no_grad():
        tensor = torch.from_numpy(volume)[None, None].to(device)
        metadata = torch.tensor([make_metadata_vector(series)], dtype=torch.float32, device=device)
        features, attentions = student(tensor, return_attn=True)
        probs = torch.sigmoid(classifier(features, metadata).float())[0].cpu().numpy()

    cam = attention_to_volume(attention_rollout(attentions), student.grid, cfg.data.target_shape)
    print('study:', study_uid[:24])
    print('top-3:', sorted(zip(LABELS, probs.round(3)), key=lambda x: -x[1])[:3])

    middle = volume.shape[0] // 2
    display = to_unit_range(volume[middle], cfg.data.norm_mode)
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
    axes[0].imshow(display, cmap='gray');                       axes[0].set_title('Lát MRI')
    axes[1].imshow(cam[middle], cmap='inferno');                axes[1].set_title('CAM')
    axes[2].imshow(display, cmap='gray')
    axes[2].imshow(cam[middle], cmap='inferno', alpha=0.45);    axes[2].set_title('Chồng lớp')
    for ax in axes:
        ax.axis('off')
    fig.tight_layout()
    plt.show()

    if mask is not None:
        from knee_mri.data.normalize import resize_volume
        resized = (resize_volume(mask.astype(np.float32), cfg.data.target_shape) > 0.5).astype(np.uint8)
        print('Trùng khớp CAM/ROI:', cam_mask_overlap(cam, resized))


## 3. AUC trên các study có nhãn gold

Đây là tín hiệu duy nhất về metric thật của cuộc thi. 58 study này được giữ làm
validation cố định, không dùng để huấn luyện.


In [ ]:
from knee_mri.inference.predictor import Predictor
from knee_mri.training.metrics import format_auc_report, macro_auc

if not checkpoint.is_file():
    print('Chưa có checkpoint.')
else:
    gold_uids, truth_rows = catalog.gold_matrix(catalog.studies_with_gold())
    print('Số study có nhãn gold:', len(gold_uids))

    predictor = Predictor.from_checkpoint(cfg, checkpoint=checkpoint, device=device)
    scores = predictor.predict_many(catalog, gold_uids, log_every=0)
    truths = np.array(truth_rows, dtype=np.float32)

    report = macro_auc(truths, scores)
    print(format_auc_report(report))

    if report.per_label:
        names = list(report.per_label)
        values = [report.per_label[n] for n in names]
        order = np.argsort(values)
        fig, ax = plt.subplots(figsize=(8, 4.5))
        ax.barh([names[i] for i in order], [values[i] for i in order], color='#55a868')
        ax.axvline(0.5, color='crimson', linestyle='--', label='ngẫu nhiên')
        ax.axvline(report.macro, color='navy', linestyle=':', label=f'macro = {report.macro:.3f}')
        ax.set_xlim(0, 1)
        ax.set_xlabel('AUC')
        ax.legend()
        ax.set_title('AUC theo từng nhãn (tập nhãn gold)')
        fig.tight_layout()
        plt.show()
